# EDA — Delay Amplification Factor Analysis (Weeks 3–4)

This notebook analyses the rotation chain data to understand:
- Distribution of the delay amplification factor α
- Airport-level and temporal patterns of high amplification
- Graph structure of the airport network
- Feature engineering insights for the Temporal GNN

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

ROOT = Path('..') 
DATA = ROOT / 'data'

chains  = pd.read_csv(DATA / 'rotation_chains.csv', low_memory=False)
states  = pd.read_csv(DATA / 'temporal_node_states.csv', low_memory=False)
nodes   = pd.read_csv(DATA / 'graph_nodes.csv', low_memory=False)
edges   = pd.read_csv(DATA / 'graph_edges.csv', low_memory=False)

chains['FlightDate'] = pd.to_datetime(chains['FlightDate'])
chains['month']      = chains['FlightDate'].dt.month
chains['hour']       = chains['dep_time_blk'].astype(str).str[:2].astype(int, errors='raise')

print(f'Rotation events : {len(chains):,}')
print(f'Airports        : {chains["turnaround_airport"].nunique()}')
print(f'Date range      : {chains["FlightDate"].min().date()} → {chains["FlightDate"].max().date()}')

## 1  α Distribution

**Formula:** `α = max(0, outbound_delay − inbound_delay) / max(1, inbound_delay)`

When a flight arrives on-time (`inbound_delay ≤ 0`), the denominator floors at 1, so any positive outbound delay directly equals α.
This means α measures both *amplification* of existing delay and *generation* of new delay.
Values > 1 indicate the outbound delay exceeds the inbound delay (net amplification).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# -- Full distribution (log scale) --
ax = axes[0]
log_alpha = np.log1p(chains['alpha'])
ax.hist(log_alpha, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
ax.set_xlabel('log(1 + α)')
ax.set_ylabel('Count')
ax.set_title('α distribution (log-scale)')

# -- Zoomed: α ≤ 5 (bulk of distribution) --
ax = axes[1]
ax.hist(chains.loc[chains['alpha'] <= 5, 'alpha'], bins=60,
        color='steelblue', edgecolor='none', alpha=0.8)
ax.axvline(0.2, color='red', lw=1.5, linestyle='--', label='Target threshold α=0.2')
ax.set_xlabel('α')
ax.set_ylabel('Count')
ax.set_title('α distribution (α ≤ 5)')
ax.legend()

# -- CDF --
ax = axes[2]
alpha_sorted = np.sort(chains['alpha'].clip(upper=50))
cdf = np.arange(1, len(alpha_sorted)+1) / len(alpha_sorted)
ax.plot(alpha_sorted, cdf, lw=1.5, color='steelblue')
ax.axvline(0.2, color='red', lw=1.5, linestyle='--', label='α=0.2')
ax.axhline(0.5, color='grey', lw=1, linestyle=':')
ax.set_xlabel('α (clipped at 50)')
ax.set_ylabel('Cumulative fraction')
ax.set_title('CDF of α')
ax.legend()

plt.tight_layout()
plt.show()

# Summary stats
print('α summary statistics')
print(chains['alpha'].describe().round(3).to_string())
print(f'\nEvents with α = 0        : {(chains["alpha"]==0).sum():,}  ({(chains["alpha"]==0).mean():.1%})')
print(f'Events with α < 0.2      : {(chains["alpha"]<0.2).sum():,}  ({(chains["alpha"]<0.2).mean():.1%})')
print(f'Events with α > 0.5      : {(chains["alpha"]>0.5).sum():,}  ({(chains["alpha"]>0.5).mean():.1%})')
print(f'Events with α > 1.0      : {(chains["alpha"]>1.0).sum():,}  ({(chains["alpha"]>1.0).mean():.1%})')

## 2  α by Inbound Delay Segment

Splitting by inbound delay shows whether the airport is absorbing existing delay or generating new delay.

In [ ]:
bins   = [-np.inf, 0, 5, 15, 30, 60, np.inf]
labels = ['Early / On-time', '1–5 min late', '6–15 min late',
          '16–30 min late', '31–60 min late', '> 60 min late']
chains['inbound_segment'] = pd.cut(chains['inbound_delay'], bins=bins, labels=labels)

seg_stats = chains.groupby('inbound_segment', observed=True)['alpha'].agg(
    count='count', mean='mean', median='median',
    pct_above_05=lambda x: (x > 0.5).mean(),
    pct_above_10=lambda x: (x > 1.0).mean(),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.bar(seg_stats['inbound_segment'], seg_stats['mean'], color='steelblue', alpha=0.8)
ax.bar(seg_stats['inbound_segment'], seg_stats['median'], color='darkorange', alpha=0.8, label='Median')
ax.set_xticklabels(seg_stats['inbound_segment'], rotation=30, ha='right')
ax.set_ylabel('α')
ax.set_title('Mean and Median α by Inbound Delay Segment')
handles = [plt.Rectangle((0,0),1,1,color='steelblue',alpha=0.8),
           plt.Rectangle((0,0),1,1,color='darkorange',alpha=0.8)]
ax.legend(handles, ['Mean', 'Median'])

ax = axes[1]
ax.bar(seg_stats['inbound_segment'], seg_stats['pct_above_05'], color='steelblue', alpha=0.8, label='α > 0.5')
ax.bar(seg_stats['inbound_segment'], seg_stats['pct_above_10'], color='crimson', alpha=0.8, label='α > 1.0')
ax.set_xticklabels(seg_stats['inbound_segment'], rotation=30, ha='right')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_title('High-Amplification Rate by Inbound Delay Segment')
ax.legend()

plt.tight_layout()
plt.show()

print(seg_stats.to_string(index=False))

## 3  Airport-Level Analysis

In [ ]:
airport_stats = chains.groupby('turnaround_airport').agg(
    n_events      = ('alpha', 'count'),
    mean_alpha    = ('alpha', 'mean'),
    median_alpha  = ('alpha', 'median'),
    pct_above_05  = ('alpha', lambda x: (x > 0.5).mean()),
    pct_above_10  = ('alpha', lambda x: (x > 1.0).mean()),
    mean_inbound  = ('inbound_delay', 'mean'),
    mean_outbound = ('outbound_delay', 'mean'),
).reset_index()

airport_stats = airport_stats.merge(
    nodes[['airport','hub_tier','total_departures']], 
    left_on='turnaround_airport', right_on='airport', how='left'
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Top-20 airports by mean_alpha (min 500 events)
top_mean = (
    airport_stats[airport_stats['n_events'] >= 500]
    .nlargest(20, 'mean_alpha')
)
colors = {'major': '#e74c3c', 'regional': '#f39c12', 'local': '#2ecc71'}
bar_colors = top_mean['hub_tier'].map(colors).fillna('grey')

ax = axes[0]
ax.barh(top_mean['turnaround_airport'], top_mean['mean_alpha'], color=bar_colors, alpha=0.85)
ax.set_xlabel('Mean α')
ax.set_title('Top 20 Airports by Mean α  (≥500 events)')
ax.invert_yaxis()
legend_handles = [plt.Rectangle((0,0),1,1,color=c,alpha=0.85) for c in colors.values()]
ax.legend(legend_handles, colors.keys(), title='Hub tier')

# Top-20 by fraction with α > 1.0
top_high = (
    airport_stats[airport_stats['n_events'] >= 500]
    .nlargest(20, 'pct_above_10')
)
bar_colors2 = top_high['hub_tier'].map(colors).fillna('grey')
ax = axes[1]
ax.barh(top_high['turnaround_airport'], top_high['pct_above_10'], color=bar_colors2, alpha=0.85)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlabel('Fraction of events with α > 1.0')
ax.set_title('Top 20 Airports by High-Amplification Rate (≥500 events)')
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 4  Temporal Patterns — Hour of Day and Day of Week

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

# Hour of day
hourly = chains.groupby('hour')['alpha'].agg(['mean', 'median', lambda x: (x > 1.0).mean()])
hourly.columns = ['mean', 'median', 'pct_high']
hourly = hourly.reset_index()

ax = axes[0]
ax.plot(hourly['hour'], hourly['mean'],   'o-', label='Mean α',   color='steelblue')
ax.plot(hourly['hour'], hourly['median'], 's--', label='Median α', color='darkorange')
ax.set_xlabel('Hour of Day (CRS Departure)')
ax.set_ylabel('α')
ax.set_title('α by Hour of Day')
ax.set_xticks(range(0, 24, 2))
ax.legend()

# Day of week  (1=Mon, 7=Sun)
dow_labels = {1:'Mon', 2:'Tue', 3:'Wed', 4:'Thu', 5:'Fri', 6:'Sat', 7:'Sun'}
daily = chains.groupby('DayOfWeek')['alpha'].agg(['mean', 'median']).reset_index()
daily['day_label'] = daily['DayOfWeek'].map(dow_labels)

ax = axes[1]
ax.bar(daily['day_label'], daily['mean'],   color='steelblue', alpha=0.8, label='Mean')
ax.bar(daily['day_label'], daily['median'], color='darkorange', alpha=0.8, label='Median')
ax.set_xlabel('Day of Week')
ax.set_ylabel('α')
ax.set_title('α by Day of Week')
ax.legend()

plt.tight_layout()
plt.show()

## 5  Monthly Trends

In [ ]:
month_labels = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
                7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

monthly = chains.groupby('month').agg(
    n_events    = ('alpha', 'count'),
    mean_alpha  = ('alpha', 'mean'),
    pct_high    = ('alpha', lambda x: (x > 1.0).mean()),
).reset_index()
monthly['month_label'] = monthly['month'].map(month_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.bar(monthly['month_label'], monthly['mean_alpha'], color='steelblue', alpha=0.85)
ax.set_xlabel('Month')
ax.set_ylabel('Mean α')
ax.set_title('Monthly Mean α (2025)')

ax = axes[1]
ax.bar(monthly['month_label'], monthly['pct_high'], color='crimson', alpha=0.85)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlabel('Month')
ax.set_ylabel('Fraction α > 1.0')
ax.set_title('Monthly High-Amplification Rate (2025)')

plt.tight_layout()
plt.show()

## 6  α Heatmap — Airport × Hour of Day

Shows which airport–time combinations are consistently high-amplification (prime candidates for larger scheduling buffers).

In [ ]:
# Restrict to major-hub airports and clip α for readability
major_airports = nodes[nodes['hub_tier'] == 'major']['airport'].tolist()

heatmap_data = (
    chains[chains['turnaround_airport'].isin(major_airports)]
    .groupby(['turnaround_airport', 'hour'])['alpha']
    .mean()
    .unstack('hour')
    .clip(upper=30)  # clip extreme outliers for visibility
)

# Sort airports by overall mean alpha
airport_order = heatmap_data.mean(axis=1).sort_values(ascending=False).index
heatmap_data = heatmap_data.loc[airport_order]

fig, ax = plt.subplots(figsize=(16, 9))
sns.heatmap(
    heatmap_data,
    cmap='YlOrRd',
    linewidths=0.3,
    ax=ax,
    cbar_kws={'label': 'Mean α (clipped at 30)'},
)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Airport')
ax.set_title('Mean α by Major Hub Airport × Hour of Day')
plt.tight_layout()
plt.show()

## 7  Turnaround Time Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Scheduled vs actual turnaround time
ax = axes[0]
ax.hist(chains['scheduled_ground_min'].clip(0, 200), bins=60,
        alpha=0.7, color='steelblue', label='Scheduled')
ax.hist(chains['actual_ground_min'].clip(0, 200), bins=60,
        alpha=0.7, color='darkorange', label='Actual')
ax.set_xlabel('Ground time (minutes)')
ax.set_ylabel('Count')
ax.set_title('Scheduled vs Actual Turnaround Time')
ax.legend()

# Buffer slack = actual - scheduled
chains['ground_slack'] = chains['actual_ground_min'] - chains['scheduled_ground_min']
ax = axes[1]
ax.hist(chains['ground_slack'].clip(-30, 60), bins=60, color='steelblue', alpha=0.8)
ax.axvline(0, color='red', lw=1.5, linestyle='--', label='Zero slack')
ax.set_xlabel('Actual − Scheduled turnaround (min)')
ax.set_ylabel('Count')
ax.set_title('Turnaround Slack Distribution')
ax.legend()

# Scheduled ground time vs α
sample = chains.sample(min(50000, len(chains)), random_state=42)
ax = axes[2]
ax.scatter(
    sample['scheduled_ground_min'].clip(0, 120),
    np.log1p(sample['alpha']),
    alpha=0.05, s=5, color='steelblue'
)
ax.set_xlabel('Scheduled ground time (min)')
ax.set_ylabel('log(1 + α)')
ax.set_title('Scheduled Turnaround vs α (sample)')

plt.tight_layout()
plt.show()

print('Scheduled ground time (min):')
print(chains['scheduled_ground_min'].describe().round(1).to_string())
print('\nActual ground time (min):')
print(chains['actual_ground_min'].describe().round(1).to_string())

## 8  Hub Tier Analysis

In [ ]:
chains_with_tier = chains.merge(
    nodes[['airport', 'hub_tier']], 
    left_on='turnaround_airport', right_on='airport', how='left'
)

tier_stats = chains_with_tier.groupby('hub_tier')['alpha'].agg(
    count='count', mean='mean', median='median',
    pct_above_05=lambda x: (x > 0.5).mean(),
    pct_above_10=lambda x: (x > 1.0).mean(),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

tier_order = ['major', 'regional', 'local']
tier_colors = ['#e74c3c', '#f39c12', '#2ecc71']

ax = axes[0]
tier_plot = tier_stats.set_index('hub_tier').reindex(tier_order)
ax.bar(tier_order, tier_plot['mean'],   color=tier_colors, alpha=0.85, label='Mean')
ax.bar(tier_order, tier_plot['median'], color='navy', alpha=0.6, label='Median', width=0.4)
ax.set_ylabel('α')
ax.set_title('α by Hub Tier')
ax.legend()

ax = axes[1]
x = np.arange(len(tier_order))
ax.bar(x - 0.2, tier_plot.loc[tier_order, 'pct_above_05'], 0.4, color='steelblue', alpha=0.85, label='α > 0.5')
ax.bar(x + 0.2, tier_plot.loc[tier_order, 'pct_above_10'], 0.4, color='crimson', alpha=0.85, label='α > 1.0')
ax.set_xticks(x)
ax.set_xticklabels(tier_order)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_title('High-Amplification Rate by Hub Tier')
ax.legend()

plt.tight_layout()
plt.show()

print(tier_stats.to_string(index=False))

## 9  Delay Cause Breakdown for High-α Events

In [ ]:
delay_cols = ['CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']

# Average delay by cause for low vs high α events
cause_low  = chains[chains['alpha'] <= 0.5][delay_cols].mean()
cause_high = chains[chains['alpha'] >  1.0][delay_cols].mean()

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(delay_cols))
ax.bar(x - 0.2, cause_low.values,  0.4, label='α ≤ 0.5 (low)', color='steelblue', alpha=0.85)
ax.bar(x + 0.2, cause_high.values, 0.4, label='α > 1.0 (high)', color='crimson', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([c.replace('Delay','').strip() for c in delay_cols], rotation=20, ha='right')
ax.set_ylabel('Mean delay (minutes)')
ax.set_title('Outbound Delay Cause Breakdown: Low vs High α Events')
ax.legend()
plt.tight_layout()
plt.show()

## 10  Graph Structure

In [ ]:
# Out-degree per node
out_deg = edges.groupby('origin')['route_frequency'].agg(out_degree='count', total_freq='sum').reset_index()
in_deg  = edges.groupby('dest')  ['route_frequency'].agg(in_degree='count').reset_index().rename(columns={'dest':'origin'})
degree  = out_deg.merge(in_deg, on='origin', how='outer').fillna(0)
degree  = degree.merge(nodes[['airport','hub_tier']], left_on='origin', right_on='airport', how='left')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ax.hist(degree['out_degree'], bins=40, color='steelblue', alpha=0.85)
ax.set_xlabel('Out-degree (number of distinct dest airports)')
ax.set_ylabel('Count')
ax.set_title('Airport Out-Degree Distribution')

ax = axes[1]
ax.scatter(degree['out_degree'], np.log1p(degree['total_freq']),
           alpha=0.5, s=20, color='steelblue')
ax.set_xlabel('Out-degree')
ax.set_ylabel('log(1 + total departures)')
ax.set_title('Out-Degree vs Route Volume')

ax = axes[2]
tier_deg = degree.groupby('hub_tier')['out_degree'].mean().reindex(['major','regional','local'])
ax.bar(tier_deg.index, tier_deg.values, color=['#e74c3c','#f39c12','#2ecc71'], alpha=0.85)
ax.set_ylabel('Mean out-degree')
ax.set_title('Mean Out-Degree by Hub Tier')

plt.tight_layout()
plt.show()

print(f'Graph summary')
print(f'  Nodes (airports) : {len(nodes):,}')
print(f'  Directed edges   : {len(edges):,}')
print(f'  Mean out-degree  : {degree["out_degree"].mean():.1f}')
print(f'  Max out-degree   : {degree["out_degree"].max():.0f}  ({degree.loc[degree["out_degree"].idxmax(), "origin"]})')
print(f'  Hub tier counts  : {nodes["hub_tier"].value_counts().to_dict()}')

## 11  Feature Engineering Summary

Based on the EDA above, the following features will be used in the Temporal GNN:

**Static node features** (from `graph_nodes.csv`):
- `hub_tier` (one-hot encoded: major / regional / local)
- `total_departures` (normalised log scale)
- `latitude`, `longitude`, `elevation_ft` (normalised)

**Static edge features** (from `graph_edges.csv`):
- `route_frequency` (log-normalised)
- `passenger_volume` (log-normalised)
- `distance_miles` (normalised)

**Dynamic node features** (from `temporal_node_states.csv`, per time-step):
- `mean_inbound_delay` — the primary delay state `d_i(t)`
- `mean_alpha` — lagged amplification factor (autoregressive)
- `event_count` — traffic volume indicator

**Time context** (added per time-step):
- `sin/cos(hour / 24 * 2π)` — cyclical hour encoding
- `sin/cos(day_of_week / 7 * 2π)` — cyclical day encoding
- `month` — seasonal context

In [ ]:
# Correlation matrix for continuous features
feature_cols = [
    'inbound_delay', 'outbound_delay', 'alpha',
    'actual_ground_min', 'scheduled_ground_min',
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'LateAircraftDelay',
]
corr = chains[feature_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
    square=True, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()